## **CAKE** (**C**onfidence in **A**ssignments via **K**-partition **E**nsembles)

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.metrics import silhouette_samples, confusion_matrix
from scipy.optimize import linear_sum_assignment
from sklearn.preprocessing import LabelEncoder

def sil_samples(X, labels, approximation=False, centers=None):
    """
    Compute silhouette scores for each point in the dataset,
    with approximate fast centroid-based computation option.
    """
    # Ensure arrays
    X = np.asarray(X)
    labels = np.asarray(labels)
    if X.ndim != 2:
       raise ValueError("X must be a 2D array of shape (n_samples, n_features).")
    if labels.ndim != 1 or labels.shape[0] != X.shape[0]:
       raise ValueError("labels must be a 1D array of length n_samples.")

    unique_labels, inv = np.unique(labels, return_inverse=True)
    k = unique_labels.size
    if k < 2:
       raise ValueError("Silhouette computation requires at least 2 clusters.")

    # Exact silhouette scores
    if approximation == False:
       silhouette_scores = silhouette_samples(X, labels=labels)
       return silhouette_scores

    # Centroid-based approximate silhouette scores
    n_samples, n_features = X.shape

    if centers is None:
       centers = np.array([X[inv == i].mean(axis=0) for i in range(k)], dtype=float)
    else:
       centers = np.asarray(centers, dtype=float)
       if centers.ndim != 2 or centers.shape[1] != n_features:
          raise ValueError(f"centers must have shape (k, d) with d={n_features}.")
       if centers.shape[0] != k:
          raise ValueError(f"centers.shape[0] must equal number of clusters k={k}.")
       if not np.array_equal(unique_labels, np.arange(k)):
          raise ValueError("When passing ndarray centers, labels must be dense 0..k-1.")

    # Squared distances to all centroids
    D_sq = euclidean_distances(X, centers, squared=True)

    # a(i): distance to own centroid
    a = np.sqrt(np.maximum(D_sq[np.arange(n_samples), inv], 0.0))

    # b(i): distance to nearest other centroid
    D_sq[np.arange(n_samples), inv] = np.inf
    b = np.sqrt(np.min(D_sq, axis=1))

    # Silhouette per point
    denom = np.maximum(np.maximum(a, b), 1e-12)
    s_point = (b - a) / denom

    # Singleton clusters -> silhouette = 0
    counts = np.bincount(inv, minlength=k).astype(int)
    s_point[counts[inv] < 2] = 0.0

    silhouette_scores = np.clip(s_point, -1.0, 1.0)

    return silhouette_scores

def sil_samples_stats(X, labels_list, approximation=False, centers_list=None):
    """
    Compute and aggregate silhouette scores across multiple clustering runs,
    returning per-sample mean and standard deviation of silhouette scores.
    """
    X = np.asarray(X)
    n_samples = X.shape[0]
    n_runs = len(labels_list)
    if n_runs < 2:
       raise ValueError("Clustering Ensemble must contain at least 2 partitions.")

    if centers_list is not None and len(centers_list) != n_runs:
       raise ValueError("centers_list must match the number of labelings in labels_list")

    # Initialize matrix to hold silhouette scores
    sil_scores = np.zeros((n_runs, n_samples), dtype=float)

    for i, labels in enumerate(labels_list):
        labels_arr = np.asarray(labels)
        centers = None
        if approximation and centers_list is not None:
           centers = centers_list[i]

        # Compute silhouette scores for this run
        sil_scores[i] = sil_samples(X, labels_arr, approximation=approximation, centers=centers)

    # Compute mean and standard deviation of sample-silhouette scores over the ensemble
    mean_sil_samples = sil_scores.mean(axis=0)
    std_sil_samples = sil_scores.std(axis=0)

    return mean_sil_samples, std_sil_samples

def align_labels(target, source):
    """
    Aligns the labels in "source" to match the labels in "target"
    using the Hungarian Algorithm based on a contingency matrix.

    Parameters:
    - target: array-like of shape (n_samples,)
              The reference label vector to align to.
    - source: array-like of shape (n_samples,)
              The label vector to be permuted for alignment.

    Returns:
    - aligned: np.ndarray of shape (n_samples,)
               The source labels, remapped to best match the target labels.
    """
    target = np.asarray(target)
    source = np.asarray(source)
    unique_target = np.unique(target)
    unique_source = np.unique(source)
    if len(unique_target) != len(unique_source):
       raise ValueError(
           f"Cannot align: Target has {len(unique_target)} clusters {unique_target.tolist()}, "
           f"but source has {len(unique_source)} clusters {unique_source.tolist()}"
       )
    # Encode labels to indices based on their unique values
    le_target = LabelEncoder().fit(unique_target)
    le_source = LabelEncoder().fit(unique_source)

    target_encoded = le_target.transform(target)
    source_encoded = le_source.transform(source)

    # Confusion matrix with indices aligned to encoded labels
    c_matrix = confusion_matrix(target_encoded, source_encoded)

    # Apply the Hungarian algorithm
    row_ind, col_ind = linear_sum_assignment(-c_matrix)

    # Create mapping from source labels to target labels
    mapping = {
        le_source.classes_[src_col]: le_target.classes_[tgt_row]
        for tgt_row, src_col in zip(row_ind, col_ind)
    }
    # Remap source labels using the mapping
    aligned = np.vectorize(mapping.get)(source)

    return aligned

def pairwise_stability(labels_runs):
    """
    Computes per-point clustering stability across multiple runs by aligning labels
    pairwise using the Hungarian algorithm to account for label permutations.

    Parameters:
    - labels_runs: list of array-like
        List of label arrays from multiple clustering runs. Each array must have the
        same number of samples and clusters (unique labels).

    Returns:
    - stability: np.ndarray of shape (n_samples,)
        Per-point stability score between 0 and 1, indicating the fraction of run-pairs
        where the point's label matches after optimal alignment.

    Notes:
    - Requires all runs to have the same number of clusters (unique labels).
    - Label alignment is done pairwise between runs using the Hungarian algorithm.
    - High stability (~1) indicates stable cluster assignments across runs.
    """
    labels_runs = [np.asarray(labels) for labels in labels_runs]
    n_runs, n_samples = len(labels_runs), len(labels_runs[0])
    if n_runs < 2:
       raise ValueError("pairwise_stability needs at least 2 runs.")

    # Counters: How many run-pairs agree per point
    agreement_counts = np.zeros(n_samples, dtype=int)
    total_pairs = 0

    # For every unique pair of runs
    for r1 in range(n_runs):
        labels_r1 = labels_runs[r1]
        for r2 in range(r1+1, n_runs):
            labels_r2 = labels_runs[r2]

            # Align labels_r2 to labels_r1 using Hungarian
            aligned_r2 = align_labels(labels_r1, labels_r2)
            matches = (labels_r1 == aligned_r2)

            # Add matches to total per-point agreement count
            agreement_counts += matches
            total_pairs += 1

    return agreement_counts/total_pairs

def cake(X, labels_list, method='product', approximation=False, centers_list=None, geom_norm='clip'):
    """
    Compute a confidence score per point for clustering ensembles, defined as:
    stability_i * geometric_stability_i or
    2 * stability_i * geometric_stability_i / [stability_i + geometric_stability_i]
    where stability is the pairwise label agreement across runs,
    and mean_sil_i, std_sil_i are the statistics from sil_samples_stats.

    Parameters:
    - X: array-like, shape (n_samples, n_features)
    - labels_list: list of array-like, each of shape (n_samples,)
        A list of cluster labelings for the same dataset X.
    - approximation: bool, default=False
        Whether to use the approximate silhouette computation.
    - centers_list: list of pd.Series or array-like, optional
        If approximation=True, an optional list of centroids for each clustering.
    - method: str, default='product'
        'product' or 'harmonic_mean' for the formulation of the CAKE scores.
    - geo_norm: str, default='clip'
        - 'affine' (default): maps geom_raw in [-1,1] to [0,1] via (x+1)/2 (preserves negatives).
        - 'clip': max(mean_sil - std_sil, 0) clipped at 1 (original behavior; discards negatives).

    Returns:
    - cake_scores: np.ndarray, shape (n_samples,)
        The CAKE scores for each point.
    - stability: np.ndarray, shape (n_samples,)
        The stability scores for each point.
    - geom_stability: np.ndarray, shape (n_samples,)
        The silhouette-based reliability scores for each point (mean_sil_i - std_sil_i).
    - summary: pd.DataFrame
        Summary of above metrics per point in X.
    """
    # Compute silhouette statistics
    mean_sil, std_sil = sil_samples_stats(X, labels_list,
                                          approximation=approximation,
                                          centers_list=centers_list)

    # Assignment stability across the ensemble
    stability = pairwise_stability(labels_list)

    # Silhouette-based stability across the ensemble
    geom_raw = mean_sil - std_sil
    if geom_norm == 'clip':
       geom_stability = np.clip(mean_sil - std_sil, 0.0, 1.0)
    else:
       geom_stability = np.clip((geom_raw + 1) / 2, 0, 1) # preserves information for negative scores

    # Calculate confidence scores
    if method == 'product':
       cake_scores = stability * geom_stability
    elif method == 'harmonic_mean':
       num = 2.0 * stability * geom_stability
       denom = np.maximum(stability + geom_stability, 1e-8)
       cake_scores = num / denom
    else:
       raise ValueError(
           f"Unknown method: {method}. Use 'product' or 'harmonic_mean'."
       )

    # Summary DataFrame
    summary = pd.DataFrame({
        'Mean Silhouette': mean_sil,
        'STD Silhouette': std_sil,
        'Geometric Stability': geom_stability,
        'Assignment Stability': stability,
        'CAKE': cake_scores
    })

    return cake_scores, stability, geom_stability, summary

In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from collections import OrderedDict
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.metrics import adjusted_rand_score
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap, Normalize
from sklearn.datasets import (
    load_iris,
    load_wine,
    load_breast_cancer,
    load_digits,
    fetch_openml,
    fetch_20newsgroups
)
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    silhouette_samples,
    adjusted_rand_score,
    normalized_mutual_info_score
)
from scipy.stats import spearmanr
from sentence_transformers import SentenceTransformer
from tensorflow.keras.datasets import fashion_mnist
from scipy.stats import t
from sklearn.metrics import f1_score, adjusted_mutual_info_score
from scipy.stats import kendalltau, spearmanr
from sklearn.mixture import GaussianMixture
from sklearn.metrics import average_precision_score, roc_auc_score

from sklearn.decomposition import PCA
from collections import Counter
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import NearestNeighbors
import time, gc

def clustering_accuracy(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    row_ind, col_ind = linear_sum_assignment(-cm)
    accuracy = cm[row_ind, col_ind].sum() / np.sum(cm)
    return accuracy

def micro_sil(X, labels, approximation=False, centers=None):
    s = sil_samples(X, labels, approximation=approximation, centers=centers)
    return float(np.mean(s))

def macro_sil(X, labels, approximation=False, centers=None):
    labels = np.asarray(labels)
    s = sil_samples(X, labels, approximation=approximation, centers=centers)
    uniq = np.unique(labels)
    cluster_means = [s[labels == lab].mean() for lab in uniq]
    return float(np.mean(cluster_means))

def consensus_medoid_majority(labels_runs):
    if not labels_runs:
        raise ValueError("labels_runs must be a non-empty list of 1D arrays.")
    labs = [np.asarray(l) for l in labels_runs]
    R = len(labs)
    n = labs[0].shape[0]
    if any(len(l) != n for l in labs):
        raise ValueError("All runs must have the same number of samples.")

    ks = {np.unique(l).size for l in labs}
    if len(ks) != 1:
        raise ValueError(f"All runs must have the same number of clusters; got {sorted(ks)}.")

    # pairwise symmetric Hamming distances after alignment
    D = np.zeros((R, R), dtype=float)
    for i in range(R):
        li = labs[i]
        for j in range(i+1, R):
            lj = labs[j]
            aligned_j = align_labels(li, lj)
            d_ij = 1.0 - np.mean(aligned_j == li)
            aligned_i = align_labels(lj, li)
            d_ji = 1.0 - np.mean(aligned_i == lj)
            D[i, j] = D[j, i] = 0.5 * (d_ij + d_ji)

    medoid = int(np.argmin(D.sum(axis=1)))
    ref = labs[medoid]
    aligned = [align_labels(ref, l) for l in labs]   # align all runs to the medoid
    A = np.vstack(aligned)                           # (R, n)

    # majority vote per point; tie -> medoid label
    z_star = np.empty(n, dtype=ref.dtype)
    for i in range(n):
        counts = Counter(A[:, i])
        top = counts.most_common(2)
        if len(top) == 1 or top[0][1] > top[1][1]:
            z_star[i] = top[0][0]
        else:
            z_star[i] = ref[i]  # tie-break to medoid label

    agree = np.mean(A == z_star, axis=0)
    return z_star, agree

## Adaptive CAKE filtering

### Datasets

In [3]:
def load_synthetic_blobs():
    X, y = make_blobs(
        n_samples=4000,
        centers=[[0, 0], [5, 5], [12, 0]],
        cluster_std=2,
        random_state=42
    )
    return X, y

def load_clusters_with_noise():
    X_clusters, y_clusters = make_blobs(
        n_samples=3000, centers=[[0, 0], [5, 5], [10, 0]],
        cluster_std=1.0, random_state=10
    )
    rng = np.random.default_rng(42)
    X_noise = np.random.uniform(low=-10, high=15, size=(1500, 2))
    y_noise = [-1] * 1500  # Label -1 for noise

    X = np.vstack([X_clusters, X_noise])
    y = np.concatenate([y_clusters, y_noise])
    return X, y

def load_square_ring():
    centers = [[-3, -3], [-3, 3], [3, -3], [3, 3]]
    stds = [1.9] * 4
    X_list, y_list = [], []

    for i, c in enumerate(centers):
        X_i, _ = make_blobs(n_samples=750, centers=[c], cluster_std=stds[i], random_state=20+i)
        X_list.append(X_i)
        y_list.extend([i]*750)

    X = np.vstack(X_list)
    y = np.array(y_list)
    return X, y

def load_synthetic_overlap():
    from sklearn.datasets import make_blobs
    import numpy as np

    centers = [[0, 0], [2, 0], [8, 8]]
    stds = [2, 2.3, 1.5]

    X_list, y_list = [], []
    for i, (c, s) in enumerate(zip(centers, stds)):
        X_i, _ = make_blobs(n_samples=1000, centers=[c], cluster_std=s, random_state=42+i)
        X_list.append(X_i)
        y_list.extend([i] * 1000)

    X = np.vstack(X_list)
    y = np.array(y_list)
    return X, y

def load_high_density_contrast():
    centers = [[0, 0], [8, 0], [4, 8]]
    stds = [0.2, 3.0, 1.0]
    sizes = [1000, 2000, 1000]

    X_list, y_list = [], []
    for i, (c, s, n) in enumerate(zip(centers, stds, sizes)):
        X_i, _ = make_blobs(n_samples=n, centers=[c], cluster_std=s, random_state=100+i)
        X_list.append(X_i)
        y_list.extend([i] * n)

    X = np.vstack(X_list)
    y = np.array(y_list)
    return X, y

def load_overlap_low_density():
    centers = [[0, 0], [4, 0], [8, 0]]
    stds = [0.4, 2.5, 0.4]
    sizes = [1000, 2000, 1000]

    X_list, y_list = [], []
    for i, (c, s, n) in enumerate(zip(centers, stds, sizes)):
        X_i, _ = make_blobs(n_samples=n, centers=[c], cluster_std=s, random_state=200+i)
        X_list.append(X_i)
        y_list.extend([i] * n)

    X = np.vstack(X_list)
    y = np.array(y_list)
    return X, y

def load_imbalanced_blobs():
    centers = [[0, 0], [5, 5], [10, 0]]
    stds = [0.3, 1.5, 2.5]
    sizes = [500, 1000, 2500]

    X_list, y_list = [], []
    for i, (c, s, n) in enumerate(zip(centers, stds, sizes)):
        X_i, _ = make_blobs(n_samples=n, centers=[c], cluster_std=s, random_state=400+i)
        X_list.append(X_i)
        y_list.extend([i] * n)

    X = np.vstack(X_list)
    y = np.array(y_list)
    return X, y

loaders_synth = {
    'S2':    load_synthetic_overlap,
    'S4':    load_square_ring,
    'S6':    load_overlap_low_density
}

In [4]:
# 20 News Groups
def load_20ng_bert(n_components=100):
    all_categories = [
            'alt.atheism',
            'comp.graphics',
            'comp.os.ms-windows.misc',
            'comp.sys.ibm.pc.hardware',
            'comp.sys.mac.hardware',
            'comp.windows.x',
            'misc.forsale',
            'rec.autos',
            'rec.motorcycles',
            'rec.sport.baseball',
            'rec.sport.hockey',
            'sci.crypt',
            'sci.electronics',
            'sci.med',
            'sci.space',
            'soc.religion.christian',
            'talk.politics.guns',
            'talk.politics.mideast',
            'talk.politics.misc',
            'talk.religion.misc']
    subset = fetch_20newsgroups(subset='all', categories=all_categories, remove=('headers', 'footers', 'quotes'))
    texts = subset.data
    labels = subset.target

    model = SentenceTransformer('all-MiniLM-L6-v2')
    X = model.encode(texts, show_progress_bar=True)

    X_scaled = StandardScaler().fit_transform(X)
    X_pca = PCA(n_components=n_components, random_state=42).fit_transform(X_scaled)

    return X_pca, labels

# Pendigits
def load_pendigits():
    data = fetch_openml('pendigits', version=1, as_frame=False)
    X = data.data
    y = LabelEncoder().fit_transform(data.target)
    return X, y

# Letter
def load_letter():
    data = fetch_openml('letter', version=1, as_frame=False)
    X = data.data
    y = LabelEncoder().fit_transform(data.target)
    X = StandardScaler().fit_transform(X)
    return X, y

# Fashion Mnist
def load_fashion_mnist_scaled():
    (X_train, y_train), _ = fashion_mnist.load_data()
    X = X_train.reshape((X_train.shape[0], -1)).astype(np.float32)
    return X, y_train

# Satimage
def load_satimage():
    X, y = fetch_openml('satimage', version=1, return_X_y=True, as_frame=False)
    y = LabelEncoder().fit_transform(y)
    pca = PCA(n_components=30, random_state=42)
    X_pca = pca.fit_transform(X)
    return X_pca, y

# Breast Cancer
def load_breast_cancer_processed():
    X, y = load_breast_cancer(return_X_y=True)
    X = StandardScaler().fit_transform(X)
    X_pca = PCA(n_components=10, random_state=42).fit_transform(X)
    return X_pca, y

loaders_real = {
    'breast_cancer':  load_breast_cancer_processed,
    'digits':         lambda: load_digits(return_X_y=True),
    '20newsgroups':   load_20ng_bert,
    'fashionmnist':   load_fashion_mnist_scaled,
    'pendigits':      load_pendigits,
    'satimage':       load_satimage
}

### Filtering

In [5]:
loaders_all = {**loaders_synth, **loaders_real}

In [6]:
results = []
ensemble_size = 20
n_trials = 10

coverage_grid = np.arange(0.40, 0.91, 0.05)
selection_restarts = 3

def mean_ci_t(scores, confidence=0.95, clip01=True):
    scores = np.array(scores, dtype=float)
    n = len(scores)
    mean = np.mean(scores)
    se = np.std(scores, ddof=1) / np.sqrt(n)
    h = se * t.ppf((1 + confidence) / 2.0, n - 1)

    lo = mean - h
    hi = mean + h

    if clip01:
        lo = max(0.0, lo)
        hi = min(1.0, hi)

    return f"{mean:.3f} [{lo:.4f}, {hi:.4f}]"


# ------------------------------------------------------------------
# Helper 1: exact top-k mask
# Keeps exactly k points, avoiding percentile/tie-size issues.
# ------------------------------------------------------------------
def topk_mask(scores, k):
    scores = np.asarray(scores)
    n = scores.shape[0]
    k = int(np.clip(k, 1, n))

    idx = np.argsort(scores)[-k:]
    mask = np.zeros(n, dtype=bool)
    mask[idx] = True
    return mask


# ------------------------------------------------------------------
# Helper 2: adaptive coverage by internal silhouette on CAKE(HM)
#
#
# 1. For each candidate retained coverage c:
#    - keep the top-c fraction by CAKE(HM)
#    - recluster that subset with KMeans
#    - compute approximate mean silhouette on the retained subset
# 2. Pick the coverage with the best internal silhouette.
#
#
# ------------------------------------------------------------------
def select_keep_n_by_internal_silhouette(
    X,
    scores,
    k,
    coverages=np.arange(0.40, 0.91, 0.05),
    selection_restarts=3,
    random_state=123
):
    X = np.asarray(X)
    scores = np.asarray(scores, dtype=float)
    n_total = X.shape[0]

    if len(scores) != n_total:
        raise ValueError("scores must have the same length as X.")

    best_keep_n = None
    best_cov = None
    best_score = -np.inf

    tried = set()

    for c in coverages:
        keep_n = int(np.clip(round(c * n_total), 2, n_total))
        if keep_n in tried:
            continue
        tried.add(keep_n)

        mask = topk_mask(scores, keep_n)
        X_sub = X[mask]

        sil_vals = []
        for r in range(selection_restarts):
            km = KMeans(
                n_clusters=k,
                init="random",
                n_init=1,
                random_state=random_state + r
            ).fit(X_sub)

            sil_vals.append(
                micro_sil(
                    X_sub,
                    km.labels_,
                    approximation=True,
                    centers=km.cluster_centers_
                )
            )

        avg_sil = float(np.mean(sil_vals))

        # tie-break: larger retained subset if the score is identical
        if (avg_sil > best_score) or (np.isclose(avg_sil, best_score) and (best_keep_n is None or keep_n > best_keep_n)):
            best_score = avg_sil
            best_keep_n = keep_n
            best_cov = keep_n / n_total

    # Fallback
    if best_keep_n is None:
        best_keep_n = int(np.clip(round(0.70 * n_total), 1, n_total))
        best_cov = best_keep_n / n_total
        best_score = np.nan

    threshold = float(np.sort(scores)[::-1][best_keep_n - 1])
    return best_keep_n, threshold, best_cov, best_score


for name, loader in loaders_all.items():
    print(f"\nProcessing dataset: {name}")
    X, y = loader()

    if not hasattr(X, "toarray"):
        X = SimpleImputer(strategy='mean').fit_transform(X)
    else:
        X = X.toarray()

    k = len(np.unique(y))
    n_total = X.shape[0]

    use_approx = True

    #print(f"\n - Creating the ensemble (R = {ensemble_size} partitions) on {name}")
    km_models = [
        KMeans(n_clusters=k, init='random', n_init=1, random_state=seed).fit(X)
        for seed in range(ensemble_size)
    ]
    ensemble = [km.labels_ for km in km_models]
    centers_list = [km.cluster_centers_ for km in km_models]

    #print(
    #    f"\n - Computing CAKE confidence estimation on {name} "
    #    f"(approximation={use_approx})"
    #)
    cake_product, stability, geom_stability, _ = cake(
        X,
        ensemble,
        method='product',
        approximation=use_approx,
        centers_list=centers_list if use_approx else None
    )

    num = 2.0 * stability * geom_stability
    denom = np.maximum(stability + geom_stability, 1e-12)
    cake_harmonic_mean = num / denom

    # --------------------------------------------------------------
    # Adaptive rule:
    # choose keep_n using internal silhouette on the CAKE(HM) ranking
    # --------------------------------------------------------------
    keep_n, cutoff_hm, selected_cov, best_internal_sil = select_keep_n_by_internal_silhouette(
        X=X,
        scores=cake_harmonic_mean,
        k=k,
        coverages=coverage_grid,
        selection_restarts=selection_restarts,
        random_state=123
    )
    coverage = keep_n / n_total

    #print(
    #    f"\n - Internal-silhouette CAKE(HM) selection on {name}: "
    #    f"cutoff={cutoff_hm:.6f} | retaining {keep_n} of {n_total} points "
    #    f"({coverage:.1%}) | sel_sil={best_internal_sil:.6f}"
    #)

    # Exact top-keep_n masks for all score-based methods
    cake_product_core_mask = topk_mask(cake_product, keep_n)
    cake_harmonic_mean_core_mask = topk_mask(cake_harmonic_mean, keep_n)
    stability_core_mask = topk_mask(stability, keep_n)
    geom_stability_core_mask = topk_mask(geom_stability, keep_n)

    # Random baseline with same retained size
    rng = np.random.RandomState(0)
    random_core_mask = np.zeros(n_total, dtype=bool)
    random_core_mask[rng.choice(n_total, size=keep_n, replace=False)] = True

    # Consensus baseline with same retained size
    z_star, agree_scores = consensus_medoid_majority(ensemble)
    consensus_core_mask = topk_mask(agree_scores, keep_n)

    for subset, mask in [
        ('full',              np.ones(n_total, dtype=bool)),
        ('random',            random_core_mask),
        ('consensus-agree',   consensus_core_mask),
        ('stability-only',    stability_core_mask),
        ('silhouette-only',   geom_stability_core_mask),
        ('CAKE-prod',         cake_product_core_mask),
        ('CAKE-hmean',        cake_harmonic_mean_core_mask),
    ]:
        X_sub, y_sub = (X, y) if subset == 'full' else (X[mask], y[mask])
        n_points = X_sub.shape[0]

        #if subset == 'full':
        #    print(f" - Evaluating {subset.upper()} subset ({n_points}/{n_total}, {n_points / n_total:.1%})")
        #else:
        #    print(
        #        f" - Evaluating {subset.upper()} subset "
        #        f"({n_points}/{n_total}, {n_points / n_total:.1%}, cutoff={cutoff_hm:.6f})"
        #    )

        ari_scores, acc_scores, ami_scores = [], [], []

        for seed in range(n_trials):
            km = KMeans(n_clusters=k, init="random", n_init=1, random_state=seed)
            labels = km.fit_predict(X_sub)

            ari_scores.append(adjusted_rand_score(y_sub, labels))
            acc_scores.append(clustering_accuracy(y_sub, labels))
            ami_scores.append(adjusted_mutual_info_score(y_sub, labels))

        results.append({
            'dataset':    name,
            'subset':     subset,
            'Retained':   f"{n_points}/{n_total} ({n_points / n_total:.1%})",
            'ARI':        mean_ci_t(ari_scores),
            'AMI':        mean_ci_t(ami_scores),
            'Accuracy':   mean_ci_t(acc_scores)
        })

all_df_adaptive = pd.DataFrame(results).set_index(['dataset', 'subset'])
#display(all_df_adaptive)

pd.set_option('display.max_rows', None)

def bold_group_col_max(df: pd.DataFrame):
    styles = pd.DataFrame('', index=df.index, columns=df.columns)
    num_re = r'([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)'
    for ds, idx in df.groupby(level=0).groups.items():
        sub = df.loc[ds]
        means = sub.apply(
            lambda col: col.astype(str).str.extract(num_re, expand=False).astype(float)
        )
        for col in df.columns:
            m = means[col].max(skipna=True)
            styles.loc[(ds, means.index[means[col] == m]), col] = 'font-weight: bold'
    return styles

styled = all_df_adaptive.style.apply(bold_group_col_max, axis=None)


Processing dataset: S2

Processing dataset: S4

Processing dataset: S6

Processing dataset: breast_cancer

Processing dataset: digits

Processing dataset: 20newsgroups


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/589 [00:00<?, ?it/s]


Processing dataset: fashionmnist

Processing dataset: pendigits

Processing dataset: satimage


In [7]:
display(styled)